In [1]:
import pandas as pd
from datasets import load_dataset
import matplotlib.pyplot as plt

c:\Users\aarni\anaconda3\envs\vero\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("Transactions2.csv")

In [3]:
import os
os.getcwd()

'c:\\Users\\aarni\\OneDrive\\Työpöytä\\Python_data_project\\vero.py'

In [29]:
import pandas as pd

# Read the CSV as string first
df = pd.read_csv("Transactions2.csv", dtype=str)

# Step 1: Replace Unicode minus with ASCII minus
df = df.replace({'−': '-'}, regex=True)

# Step 2: Replace comma decimal with dot
numeric_cols = ["Kurssi", "Markkina-arvo", "Value EUR", "AutoFX Fee",
                "Transaction and/or third party fees EUR", "Kokonaissumma EUR", "Quantity"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].str.replace(",", ".").astype(float)

# Step 3: Optional - fix date and time
df['Päiväys'] = pd.to_datetime(df['Päiväys'], format="%d-%m-%Y")
df['Datetime'] = pd.to_datetime(df['Päiväys'].dt.strftime('%Y-%m-%d') + ' ' + df['Aika'])

# Now negative values and decimals should read correctly
print(df[['Tuote', 'Quantity', 'Kurssi', 'Value EUR']])

                                                 Tuote  Quantity     Kurssi  \
0                             ROLLS-ROYCE HOLDINGS PLC      70.0  1124.5000   
1    VANGUARD FTSE ALL-WORLD UCITS - (USD) ACCUMULA...      -6.0   142.9200   
2                                      DRONESHIELD LTD     220.0     1.5660   
3                                            AIRBUS SE       8.0   204.4000   
4                                            ADOBE INC      -5.0   337.1000   
..                                                 ...       ...        ...   
181                                          APPLE INC       8.0   225.2600   
182                                         INVISIO AB     -30.0   329.0000   
183                                         INVISIO AB     -30.0   329.0000   
184                                      SONO GROUP NV   -1000.0     0.0951   
185                                      SONO GROUP NV      13.0     7.1325   

     Value EUR  
0      -896.22  
1       857.52  


In [9]:
df = pd.read_csv("Transactions_Clean.csv", dtype=str)

In [12]:
df.head()

,Datetime,Symbol,ISIN,Reference exchange,Venue,Quantity,Price,Value EUR,AutoFX Fee,Fees EUR,Total EUR,Order ID
0,2025-10-10 21:31:00,ADOBE INC,US00724F1012,NDQ,ARCX,-2.0,337.70,581.59,-1.45,-2.0,579.59,2399e3c1-7b50-4f2c-8055-fd999183fe39
1,2025-10-14 15:58:00,ADOBE INC,US00724F1012,NDQ,CDED,4.0,335.27,-1158.42,-2.90,-2.0,-1160.42,884824c3-045e-422a-a913-19d165764712
2,2025-11-12 18:51:00,ADOBE INC,US00724F1012,NDQ,XNAS,-5.0,337.10,1454.03,-3.64,-2.0,1452.03,NaN
3,2025-11-05 14:53:00,ADR ON BANCO BILBAO VIZCAYA ARGENTARIA SA,US05946K1016,FRA,FRAB,100.0,17.50,-1750.00,0.00,-6.0,-1756.00,NaN
4,2025-11-11 10:00:00,ADR ON BANCO BILBAO VIZCAYA ARGENTARIA SA,US05946K1016,FRA,FRAB,-100.0,18.20,1820.00,0.00,-6.0,1814.00,NaN


In [26]:
import pandas as pd

# 1. Load CSV as string to handle weird characters
df = pd.read_csv("Transactions2.csv", dtype=str)

# 2. Replace Unicode minus with ASCII minus
df = df.replace({'−': '-'}, regex=True)

# 3. Replace comma with dot for decimals
numeric_cols = ["Kurssi", "Markkina-arvo", "Value EUR", "AutoFX Fee",
                "Transaction and/or third party fees EUR", "Kokonaissumma EUR", "Quantity"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].str.replace(",", ".").astype(float)

# 4. Fix dates and combine date/time
df['Päiväys'] = pd.to_datetime(df['Päiväys'], format="%d-%m-%Y")
df['Datetime'] = pd.to_datetime(df['Päiväys'].dt.strftime('%Y-%m-%d') + ' ' + df['Aika'])

# 5. Select relevant columns for a Transactions CSV
transactions = df[[
    'Datetime',        # full timestamp
    'Tuote',           # stock/ETF name
    'ISIN',            # ISIN
    'Reference exchange',
    'Venue',
    'Quantity',
    'Kurssi',          # price
    'Value EUR',
    'AutoFX Fee',
    'Transaction and/or third party fees EUR',
    'Kokonaissumma EUR',
    'Order ID'
]]

# 6. Rename columns to English (optional, easier for Python processing)
transactions = transactions.rename(columns={
    "Tuote": "Symbol",
    "Kurssi": "Price",
    "Kokonaissumma EUR": "Total EUR",
    "Transaction and/or third party fees EUR": "Fees EUR",
})

# 7. Sort by symbol and datetime for FIFO
transactions = transactions.sort_values(['Symbol', 'Datetime'])

# 8. Export cleaned Transactions CSV
transactions.to_csv("Transactions_Clean.csv", index=False, float_format="%.2f")

print("Transactions CSV created: Transactions_Clean.csv")

Transactions CSV created: Transactions_Clean.csv


In [22]:
# 1. Load your cleaned CSV
df = pd.read_csv("Transactions_Clean.csv", parse_dates=["Datetime"])

# 2. Ensure numeric columns
df["Quantity"] = pd.to_numeric(df["Quantity"], errors='coerce')
df["Total EUR"] = pd.to_numeric(df["Total EUR"], errors='coerce')

# 3. Create Buy/Sell columns
df["Buy"] = df["Quantity"].apply(lambda x: x if x > 0 else 0)
df["Sell"] = df["Quantity"].apply(lambda x: -x if x < 0 else 0)  # Make sell positive

# 4. Optional: keep Total EUR as profit/loss
df["Total Profit"] = df["Total EUR"]

# 5. Save new CSV
df.to_csv("Transactions_Buy_Sell.csv", index=False)

print("Transactions_Buy_Sell.csv created successfully!")

Transactions_Buy_Sell.csv created successfully!


In [27]:
df = pd.read_csv("Transactions_Buy_Sell.csv", dtype=str)

In [28]:
pd.read_csv("Transactions_Buy_Sell.csv", dtype=str)

,Datetime,Symbol,ISIN,Reference exchange,Venue,Quantity,Price,Value EUR,AutoFX Fee,Fees EUR,Total EUR,Order ID,Buy,Sell,Total Profit
0,2025-10-10 21:31:00,ADOBE INC,US00724F1012,NDQ,ARCX,-2.0,337.7,581.59,-1.45,-2.0,579.59,2399e3c1-7b50-4f2c-8055-fd999183fe39,0.0,2.0,579.59
1,2025-10-14 15:58:00,ADOBE INC,US00724F1012,NDQ,CDED,4.0,335.27,-1158.42,-2.9,-2.0,-1160.42,884824c3-045e-422a-a913-19d165764712,4.0,0.0,-1160.42
2,2025-11-12 18:51:00,ADOBE INC,US00724F1012,NDQ,XNAS,-5.0,337.1,1454.03,-3.64,-2.0,1452.03,NaN,0.0,5.0,1452.03
3,2025-11-05 14:53:00,ADR ON BANCO BILBAO VIZCAYA ARGENTARIA SA,US05946K1016,FRA,FRAB,100.0,17.5,-1750.0,0.0,-6.0,-1756.0,NaN,100.0,0.0,-1756.0
4,2025-11-11 10:00:00,ADR ON BANCO BILBAO VIZCAYA ARGENTARIA SA,US05946K1016,FRA,FRAB,-100.0,18.2,1820.0,0.0,-6.0,1814.0,NaN,0.0,100.0,1814.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,2025-03-13 10:42:00,VESTAS WIND SYSTEMS A/S,DK0061539921,OMK,MESI,47.0,104.3,-657.18,-1.64,NaN,-657.18,fdb25613-29a5-4e28-ad17-7fdec2d0efe3,47.0,0.0,-657.18
182,2025-03-13 10:42:00,VESTAS WIND SYSTEMS A/S,DK0061539921,OMK,XCSE,48.0,104.3,-671.17,-1.68,-4.9,-676.07,fdb25613-29a5-4e28-ad17-7fdec2d0efe3,48.0,0.0,-676.07
183,2025-07-28 09:00:00,VESTAS WIND SYSTEMS A/S,DK0061539921,OMK,XCSE,-95.0,127.5,1622.86,-4.06,-4.9,1617.96,cdac18f3-c40a-4691-baf1-274ab554ff9b,0.0,95.0,1617.96
184,2025-07-28 09:17:00,VESTAS WIND SYSTEMS A/S,DK0061539921,OMK,MESI,100.0,126.75,-1698.25,-4.25,-4.9,-1703.15,3f2f353c-349f-4416-875c-cb478a231eb1,100.0,0.0,-1703.15
